# PySpark reference

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference.ipynb)

The PySpark API on one page, for revision and quick lookup. Each section links to the lesson that explains and runs
it. Examples assume `from pyspark.sql import functions as F, Window` and a DataFrame `df`.

*Last reviewed: November 2026, against Apache Spark 4.0.*

## Session and configuration

| task | code |
| --- | --- |
| create or get a session | `SparkSession.builder.master("local[2]").appName("job").getOrCreate()` |
| connect to a remote server | `SparkSession.builder.remote("sc://host:15002").getOrCreate()` |
| read or change a runtime setting | `spark.conf.get("spark.sql.shuffle.partitions")`, `spark.conf.set(...)` |
| partitions of a DataFrame | `df.rdd.getNumPartitions()`, `F.spark_partition_id()` |
| plans | `df.explain()`, `df.explain("extended")`, `df.explain("formatted")`, `df.explain("cost")` |
| cache | `df.cache()`, `df.persist(StorageLevel.MEMORY_AND_DISK)`, `df.unpersist()` |
| cut lineage | `spark.sparkContext.setCheckpointDir(path)`, `df.checkpoint()`, `df.localCheckpoint()` |

Defaults worth knowing: `spark.sql.shuffle.partitions` 200, `spark.sql.autoBroadcastJoinThreshold` 10 MB,
`spark.sql.files.maxPartitionBytes` 128 MB, AQE on, ANSI mode on (Spark 4). [Lesson 1](pyspark-01-how-spark-runs.ipynb)

## Creating and inspecting DataFrames

| task | code |
| --- | --- |
| from rows with a DDL schema | `spark.createDataFrame(rows, "id int, name string")` |
| a range of ids | `spark.range(10)` |
| schema and types | `df.printSchema()`, `df.columns`, `df.dtypes`, `df.schema` |
| look at rows | `df.show(5, truncate=False)`, `df.show(vertical=True)` |
| statistics | `df.describe()`, `df.summary("count", "min", "50%", "max")` |
| to the driver | `df.collect()`, `df.take(5)`, `df.first()`, `df.toPandas()` (small results only) |
| sample and split | `df.sample(0.1, seed=7)`, `df.randomSplit([0.8, 0.2], seed=7)` |

[Lesson 2](pyspark-02-dataframe-essentials.ipynb)

## Selecting, filtering and reshaping

| task | code |
| --- | --- |
| select and compute | `df.select("a", F.col("b"), (F.col("c") * 2).alias("c2"))`, `df.selectExpr("a", "c * 2 AS c2")` |
| add or replace columns | `df.withColumn("x", expr)`, `df.withColumns({"x": e1, "y": e2})` |
| rename and drop | `df.withColumnRenamed("a", "b")`, `df.withColumnsRenamed({...})`, `df.drop("a")` |
| filter | `df.where((F.col("a") > 1) & F.col("b").isin("x", "y"))`, `between`, `like`, `rlike`, `isNull` |
| sort and limit | `df.orderBy(F.desc("a"), F.col("b").asc_nulls_last())`, `df.limit(10)` |
| duplicates | `df.distinct()`, `df.dropDuplicates(["key"])` |
| stack rows | `df.unionByName(other, allowMissingColumns=True)` (never plain `union` across sources) |
| set operations | `intersect`, `subtract`, `exceptAll`, `intersectAll` |

Traps: put each comparison in parentheses; a comparison with null is never true; sort before `limit`.
[Lesson 2](pyspark-02-dataframe-essentials.ipynb)

## Types, nulls and functions

| area | functions |
| --- | --- |
| casting | `col.cast("decimal(12,2)")`, `F.expr("try_cast(x AS INT)")`, `F.try_to_number`, `F.try_divide`, `F.try_to_timestamp` |
| nulls | `isNull`, `eqNullSafe`, `F.coalesce`, `F.nullif`, `df.fillna({...})`, `df.dropna(subset=[...])` |
| conditions | `F.when(c, v).when(...).otherwise(d)`, `F.greatest`, `F.least` |
| strings | `trim`, `lower`, `upper`, `initcap`, `concat_ws`, `substring`, `lpad`, `split`, `regexp_extract`, `regexp_replace`, `translate` |
| numbers | `round`, `bround`, `ceil`, `floor`, `abs`, `pow`; `7 div 2` in SQL for integer division |
| dates | `to_date(c, "dd/MM/yyyy")`, `to_timestamp`, `date_format`, `date_add`, `add_months`, `datediff`, `date_trunc`, `last_day`, `dayofweek` |
| time zones | `F.from_utc_timestamp(ts, "America/Chicago")`, `spark.sql.session.timeZone`, `timestamp_ntz` |
| hashes and ids | `sha2(c, 256)`, `xxhash64(...)`, `monotonically_increasing_id()` (not stable), `uuid()` (not stable) |

Money is `decimal`; `F.lit(0.2)` is a `double`. [Lesson 3](pyspark-03-types-functions-and-nulls.ipynb)

## Reading and writing

| task | code |
| --- | --- |
| CSV | `spark.read.schema(s).option("header", True).option("sep", ";").option("nullValue", "-").option("dateFormat", "dd/MM/yyyy").csv(path)` |
| JSON | `spark.read.schema(s).json(path)`; one document per file: `.option("multiLine", True)` |
| Parquet, ORC | `spark.read.parquet(path)`, `spark.read.orc(path)`; combine versions: `.option("mergeSchema", True)` |
| bad records | `.option("mode", "PERMISSIVE").option("columnNameOfCorruptRecord", "_corrupt_record")`, `DROPMALFORMED`, `FAILFAST` |
| lineage | `F.col("_metadata.file_name")`, `_metadata.file_path`, `_metadata.file_modification_time` |
| JDBC | `spark.read.format("jdbc").option("url", ...).option("dbtable", ...).option("partitionColumn", "id").option("lowerBound", 1).option("upperBound", n).option("numPartitions", 16).load()` |
| write | `df.write.mode("overwrite").partitionBy("order_date").parquet(path)`; modes `append`, `overwrite`, `ignore`, `errorifexists` |
| file count | `df.repartition(n)`, `df.coalesce(n)`, `.option("maxRecordsPerFile", n)` |
| replace touched partitions only | `spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")` |

[Lesson 4](pyspark-04-reading-and-writing-data.ipynb), [lesson 10](pyspark-10-batch-pipelines.ipynb)

## Spark SQL and the catalog

| task | code |
| --- | --- |
| name a DataFrame | `df.createOrReplaceTempView("orders")`, `createOrReplaceGlobalTempView` (read as `global_temp.name`) |
| run SQL | `spark.sql("SELECT ...")` |
| parameters | `spark.sql("... WHERE status = :s", args={"s": "paid"})`, `spark.sql("SELECT * FROM {t}", t=df)` |
| tables | `df.write.saveAsTable("db.t")`; external: `.option("path", p).saveAsTable(...)` |
| catalog | `spark.catalog.listTables("db")`, `listColumns("db.t")`, `tableExists("db.t")`, `currentDatabase()` |
| DDL | `CREATE TABLE ... USING parquet PARTITIONED BY (...)`, `INSERT OVERWRITE ... PARTITION (...)`, `ALTER TABLE ... ADD COLUMNS`, `DESCRIBE EXTENDED` |
| functions | `SHOW FUNCTIONS LIKE 'array_*'`, `DESCRIBE FUNCTION EXTENDED name` |
| table formats | `MERGE INTO`, `UPDATE`, `DELETE`, `VERSION AS OF n`, `DESCRIBE HISTORY` (Delta Lake, Iceberg) |

[Lesson 5](pyspark-05-spark-sql-and-the-catalog.ipynb)

## Joins

| `how` | keeps |
| --- | --- |
| `inner` | matches only |
| `left` | every left row |
| `full` | every row from both sides |
| `left_semi` | left rows with a match, no duplication, left columns only |
| `left_anti` | left rows without a match |
| `cross` | every pair (`df.crossJoin(other)`) |

| task | code |
| --- | --- |
| join on names (one key column) | `a.join(b, ["k1", "k2"], "left")` |
| null-safe keys | `a.join(b, a["k"].eqNullSafe(b["k"]))` |
| range or as-of | `a.join(b, (a.k == b.k) & (a.ts >= b.valid_from) & (a.ts < b.valid_to), "left")` |
| broadcast | `a.join(F.broadcast(b), "k")` |
| strategy hints | `b.hint("shuffle_hash")`, `b.hint("merge")` |

Check key uniqueness before joining and row counts after. [Lesson 6](pyspark-06-joins.ipynb)

## Aggregations and windows

| task | code |
| --- | --- |
| group | `df.groupBy("k").agg(F.count("*"), F.sum("x"), F.countDistinct("y"), F.collect_set("z"))` |
| conditional | `F.sum(F.when(cond, F.col("x")))`, `F.count_if(cond)`, SQL `sum(x) FILTER (WHERE ...)` |
| approximate | `F.approx_count_distinct(c, rsd=0.01)`, `F.percentile_approx(c, 0.95)`, exact `F.median(c)` |
| pivot and back | `df.groupBy("k").pivot("status", ["a", "b"]).agg(...)`, `df.unpivot("k", ["a", "b"], "status", "value")` |
| subtotals | `rollup("a", "b")`, `cube("a", "b")`, SQL `GROUPING SETS`, label with `F.grouping_id()` |
| window | `w = Window.partitionBy("k").orderBy("ts")` |
| ranking | `F.row_number().over(w)`, `F.rank()`, `F.dense_rank()`, `F.ntile(4)` |
| neighbours | `F.lag("x").over(w)`, `F.lead("x").over(w)` |
| frames | `w.rowsBetween(Window.unboundedPreceding, Window.currentRow)`, `w.rangeBetween(-7 * 86400, 0)` over a numeric order |
| latest per key | `row_number()` over `orderBy(F.desc("version"), F.desc("ingested_at"))`, keep `rn = 1` |

The default frame with `orderBy` ends at the current row, so `last` returns the current row. [Lesson 7](pyspark-07-aggregations-and-windows.ipynb)

## Nested and semi-structured data

| task | code |
| --- | --- |
| build | `F.struct(...)`, `F.array(...)`, `F.create_map(...)`, `F.collect_list(F.struct(...))` |
| read structs | `F.col("s.field")`, `df.select("s.*")`, `col.withField(...)`, `col.dropFields(...)` |
| arrays | `size`, `element_at(a, -1)`, `array_contains`, `sort_array`, `array_distinct`, `arrays_zip`, `array_join` |
| rows from arrays | `explode_outer` (keeps null and empty), `posexplode_outer`, `inline` |
| higher-order | `F.transform(a, f)`, `F.filter(a, f)`, `F.exists`, `F.forall`, `F.aggregate(a, start, f)` (cast start and step) |
| maps | `m["key"]`, `map_keys`, `map_values`, `map_filter`, `transform_values` |
| JSON | `F.from_json(c, schema)`, `F.to_json(c)`, `F.schema_of_json(sample)`, `F.get_json_object(c, "$.a")` |
| VARIANT | `F.parse_json(c)`, `F.variant_get(v, "$.a", "int")`, `F.try_variant_get`, `F.schema_of_variant` |

[Lesson 8](pyspark-08-nested-and-semi-structured-data.ipynb)

## Python in Spark

| kind | code | runs |
| --- | --- | --- |
| row UDF | `F.udf(fn, "string")`, `spark.udf.register("name", fn, "string")` | once per row, pickled |
| Arrow UDF | `F.udf(fn, "string", useArrow=True)` | once per row, Arrow transfer |
| pandas UDF | `F.pandas_udf(fn, "double")` with `pd.Series` type hints | once per batch, vectorized |
| grouped map | `df.groupBy("k").applyInPandas(fn, schema)` | once per group |
| map over batches | `df.mapInPandas(fn, schema)` | once per batch, any row count |
| table function | `F.udtf(cls, returnType="a string, b int")` | many rows per input |
| custom source | subclass `DataSource` and `DataSourceReader`, `spark.dataSource.register(cls)` | one task per partition |

Declare return types exactly (mismatches become null). Prefer built-ins first. [Lesson 9](pyspark-09-python-udfs-and-pandas.ipynb)

## Structured Streaming

| task | code |
| --- | --- |
| read | `spark.readStream.schema(s).option("maxFilesPerTrigger", 1).json(path)`; Kafka: `.format("kafka")` |
| watermark | `.withWatermark("event_ts", "10 minutes")` |
| dedupe | `.dropDuplicatesWithinWatermark(["event_id"])` |
| windows | `F.window("ts", "5 minutes")`, `F.window(ts, "10 minutes", "5 minutes")`, `F.session_window("ts", "10 minutes")` |
| write | `.writeStream.outputMode("update").option("checkpointLocation", p)` (or `append`, `complete`) |
| sinks | `.toTable("t")`, `.foreachBatch(fn)`, `.format("memory").queryName("q")` |
| triggers | `.trigger(availableNow=True)`, `.trigger(processingTime="1 minute")` |
| monitor | `query.lastProgress`, `query.recentProgress`, `query.awaitTermination(timeout)`, `query.stop()` |

One checkpoint per query; idempotent sinks; watermarks on both sides of stream-stream joins. [Lesson 11](pyspark-11-structured-streaming.ipynb)

## Performance settings and levers

| symptom | lever |
| --- | --- |
| one task much slower than the rest | AQE skew join (`spark.sql.adaptive.skewJoin.enabled`), salting, fix the hot key |
| large shuffle | broadcast the small side, filter and aggregate earlier, fewer wide operations |
| spill | more partitions (`advisoryPartitionSizeInBytes`), then more memory |
| many tiny tasks or files | fewer partitions, AQE coalescing, compaction (`OPTIMIZE`) |
| full scans | partition pruning, pushdown-friendly filters, dynamic partition pruning, clustering |
| repeated large joins on one key | bucketing (`bucketBy`) or table clustering |
| driver out of memory | no large `collect`/`toPandas`, smaller broadcasts |
| Python bottleneck | built-ins, then pandas or Arrow UDFs |

Shuffle partitions: `ceil(shuffle_bytes / ~128 MB)`. Executors: 3 to 5 cores and 8 to 32 GB each, plus overhead.
[Lesson 12](pyspark-12-internals-and-monitoring.ipynb), [lesson 13](pyspark-13-performance-tuning.ipynb)

## Production checklist

- Logic in pure functions, I/O in its own module, a parameterized entry point. [Lesson 14](pyspark-14-production-pyspark.ipynb)
- Business dates and paths are parameters; secrets come from a secret store.
- Tests with one shared local session, comparing rows in any order and the schema.
- Writes are idempotent: dynamic partition overwrite or a merge by key and version.
- Bad rows are quarantined with a reason; a gate blocks publishing on hard rules; counts and totals are reconciled.
- Row counts and durations are logged and stored per run; alerts fire on unusual output.
- Scheduled jobs run in cluster mode with dependencies shipped explicitly.